In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../ml-data/processed/opd_wait_time.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH.resolve())
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

In [ ]:
print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
print(df.isna().sum())

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

print("\nWAIT-TIME SUMMARY")
display(df["ActualWaitMinutes"].describe())

In [ ]:
import matplotlib.pyplot as plt

target = df["ActualWaitMinutes"]

q1 = target.quantile(0.25)
q3 = target.quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + (1.5 * iqr)

print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("IQR:", round(iqr, 2))
print("Statistical upper limit:", round(upper_limit, 2))
print("Rows above upper limit:", (target > upper_limit).sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(target, bins=40, color="#1677c8", edgecolor="white")
axes[0].set_title("OPD Waiting-Time Distribution")
axes[0].set_xlabel("Waiting time in minutes")
axes[0].set_ylabel("Number of visits")

axes[1].boxplot(target, vert=False)
axes[1].set_title("Waiting-Time Box Plot")
axes[1].set_xlabel("Waiting time in minutes")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["ActualWaitMinutes"])
y = df["ActualWaitMinutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Input features:", X_train.shape[1])
print("Training target mean:", round(y_train.mean(), 2))
print("Testing target mean:", round(y_test.mean(), 2))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_columns = X_train.select_dtypes(
    include=["object", "string", "bool"]
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_columns,
        ),
    ]
)

print("Categorical columns:")
print(categorical_columns)

print("\nNumeric columns:")
print(numeric_columns)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = mean_squared_error(
    y_test,
    baseline_predictions
) ** 0.5
baseline_r2 = r2_score(y_test, baseline_predictions)

print("BASELINE RESULTS")
print("MAE:", round(baseline_mae, 2), "minutes")
print("RMSE:", round(baseline_rmse, 2), "minutes")
print("R²:", round(baseline_r2, 4))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

random_forest_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=16,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_model.fit(X_train, y_train)

train_predictions = random_forest_model.predict(X_train)
test_predictions = random_forest_model.predict(X_test)

train_mae = mean_absolute_error(y_train, train_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = mean_squared_error(y_test, test_predictions) ** 0.5
test_r2 = r2_score(y_test, test_predictions)

print("RANDOM FOREST RESULTS")
print("Training MAE:", round(train_mae, 2), "minutes")
print("Testing MAE:", round(test_mae, 2), "minutes")
print("Testing RMSE:", round(test_rmse, 2), "minutes")
print("Testing R²:", round(test_r2, 4))

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor

gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessing", clone(preprocessor)),
        (
            "model",
            HistGradientBoostingRegressor(
                loss="absolute_error",
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=20,
                min_samples_leaf=20,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ]
)

gradient_boosting_model.fit(X_train, y_train)

gb_train_predictions = gradient_boosting_model.predict(X_train)
gb_test_predictions = gradient_boosting_model.predict(X_test)

gb_train_mae = mean_absolute_error(y_train, gb_train_predictions)
gb_test_mae = mean_absolute_error(y_test, gb_test_predictions)
gb_test_rmse = mean_squared_error(y_test, gb_test_predictions) ** 0.5
gb_test_r2 = r2_score(y_test, gb_test_predictions)

print("GRADIENT BOOSTING RESULTS")
print("Training MAE:", round(gb_train_mae, 2), "minutes")
print("Testing MAE:", round(gb_test_mae, 2), "minutes")
print("Testing RMSE:", round(gb_test_rmse, 2), "minutes")
print("Testing R²:", round(gb_test_r2, 4))

In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

models = {
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model,
}

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
    )

    mean_mae = -scores["test_mae"].mean()
    mean_rmse = -scores["test_rmse"].mean()
    mean_r2 = scores["test_r2"].mean()

    print(f"\n{model_name}")
    print("Cross-validation MAE:", round(mean_mae, 2))
    print("Cross-validation RMSE:", round(mean_rmse, 2))
    print("Cross-validation R²:", round(mean_r2, 4))

In [ ]:
from pathlib import Path
import joblib

final_model = gradient_boosting_model
final_model.fit(X, y)

MODEL_DIRECTORY = Path("../ml-models")
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIRECTORY / "opd_wait_time_model.joblib"
joblib.dump(final_model, MODEL_PATH)

print("Final model:", type(final_model.named_steps["model"]).__name__)
print("Saved at:", MODEL_PATH.resolve())
print(
    "Model size:",
    round(MODEL_PATH.stat().st_size / (1024 * 1024), 2),
    "MB",
)

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

sample_features = X_test.iloc[[0]].copy()
sample_index = sample_features.index[0]

actual_wait = float(y_test.loc[sample_index])
predicted_wait = float(
    loaded_model.predict(sample_features)[0]
)

# Waiting time negative nahi dikhni chahiye
predicted_wait = max(0.0, predicted_wait)

display(sample_features)

print("Actual waiting time:", round(actual_wait, 2), "minutes")
print("Predicted waiting time:", round(predicted_wait, 2), "minutes")
print(
    "Absolute error:",
    round(abs(actual_wait - predicted_wait), 2),
    "minutes",
)

In [ ]:
def estimate_opd_wait(
    department,
    appointment_type,
    arrival_method,
    triage_category,
    occupancy_rate,
    providers_on_shift,
    nurses_on_shift,
    staff_patient_ratio,
    arrival_hour,
    arrival_day,
    is_weekend,
):
    input_data = pd.DataFrame(
        [
            {
                "Department": department,
                "AppointmentType": appointment_type,
                "ArrivalMethod": arrival_method,
                "TriageCategory": triage_category,
                "FacilityOccupancyRate": occupancy_rate,
                "ProvidersOnShift": providers_on_shift,
                "NursesOnShift": nurses_on_shift,
                "StaffToPatientRatio": staff_patient_ratio,
                "ArrivalHour": arrival_hour,
                "ArrivalDayOfWeek": arrival_day,
                "IsWeekend": is_weekend,
            }
        ]
    )

    prediction = float(loaded_model.predict(input_data)[0])

    # Training dataset ka minimum wait one minute hai
    return round(max(1.0, prediction), 1)

In [ ]:
estimated_minutes = estimate_opd_wait(
    department="Cardiology",
    appointment_type="Urgent Care",
    arrival_method="Scheduled",
    triage_category="Non-urgent",
    occupancy_rate=0.66,
    providers_on_shift=8,
    nurses_on_shift=12,
    staff_patient_ratio=0.21,
    arrival_hour=16,
    arrival_day=6,
    is_weekend=True,
)

print("SmartCare estimated wait:", estimated_minutes, "minutes")
print("Patient message: Expected consultation in approximately",
      estimated_minutes, "minutes.")


In [ ]:
low_load_wait = estimate_opd_wait(
    department="Cardiology",
    appointment_type="Urgent Care",
    arrival_method="Scheduled",
    triage_category="Non-urgent",
    occupancy_rate=0.30,
    providers_on_shift=8,
    nurses_on_shift=14,
    staff_patient_ratio=0.40,
    arrival_hour=16,
    arrival_day=2,
    is_weekend=False,
)

high_load_wait = estimate_opd_wait(
    department="Cardiology",
    appointment_type="Urgent Care",
    arrival_method="Scheduled",
    triage_category="Non-urgent",
    occupancy_rate=0.90,
    providers_on_shift=3,
    nurses_on_shift=5,
    staff_patient_ratio=0.10,
    arrival_hour=16,
    arrival_day=2,
    is_weekend=False,
)

print("Low-load estimated wait:", low_load_wait, "minutes")
print("High-load estimated wait:", high_load_wait, "minutes")
print(
    "Difference:",
    round(high_load_wait - low_load_wait, 1),
    "minutes",
)

In [ ]:
numeric_analysis_columns = [
    "FacilityOccupancyRate",
    "ProvidersOnShift",
    "NursesOnShift",
    "StaffToPatientRatio",
    "ArrivalHour",
    "ArrivalDayOfWeek",
    "ActualWaitMinutes",
]

correlations = (
    df[numeric_analysis_columns]
    .corr()["ActualWaitMinutes"]
    .sort_values(ascending=False)
)

print("CORRELATION WITH WAITING TIME")
display(correlations)

In [ ]:
occupancy_analysis = (
    df.assign(
        OccupancyBand=pd.cut(
            df["FacilityOccupancyRate"],
            bins=[0, 0.40, 0.60, 0.80, 1.00],
            labels=["Low", "Medium", "High", "Very High"],
            include_lowest=True,
        )
    )
    .groupby("OccupancyBand", observed=True)["ActualWaitMinutes"]
    .agg(["count", "mean", "median"])
)

print("WAIT BY OCCUPANCY BAND")
display(occupancy_analysis)

In [ ]:
numeric_columns = [
    "FacilityOccupancyRate",
    "ProvidersOnShift",
    "NursesOnShift",
    "StaffToPatientRatio",
    "ArrivalHour",
    "ArrivalDayOfWeek",
    "ActualWaitMinutes",
]

correlations = (
    df[numeric_columns]
    .corr()["ActualWaitMinutes"]
    .sort_values(ascending=False)
)

print("CORRELATION WITH ACTUAL WAITING TIME")
display(correlations)

In [ ]:
categorical_columns = [
    "Department",
    "AppointmentType",
    "ArrivalMethod",
    "TriageCategory",
    "IsWeekend",
]

for column in categorical_columns:
    print(f"\nWAIT TIME BY {column.upper()}")

    result = (
        df.groupby(column)["ActualWaitMinutes"]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
        .round(2)
    )

    display(result)

In [ ]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

core_features = [
    "Department",
    "TriageCategory",
]

X_core = df[core_features]
y_core = df["ActualWaitMinutes"]

X_core_train, X_core_test, y_core_train, y_core_test = train_test_split(
    X_core,
    y_core,
    test_size=0.20,
    random_state=42,
)

core_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            core_features,
        )
    ],
    remainder="drop",
)

core_wait_model = Pipeline(
    steps=[
        ("preprocessor", core_preprocessor),
        (
            "model",
            HistGradientBoostingRegressor(
                loss="absolute_error",
                learning_rate=0.05,
                max_iter=250,
                max_leaf_nodes=12,
                min_samples_leaf=25,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ]
)

core_wait_model.fit(X_core_train, y_core_train)

core_train_predictions = core_wait_model.predict(X_core_train)
core_test_predictions = core_wait_model.predict(X_core_test)

print("CORE BASE-WAIT MODEL")
print(
    "Training MAE:",
    round(mean_absolute_error(y_core_train, core_train_predictions), 2),
    "minutes",
)
print(
    "Testing MAE:",
    round(mean_absolute_error(y_core_test, core_test_predictions), 2),
    "minutes",
)
print(
    "Testing RMSE:",
    round(np.sqrt(mean_squared_error(y_core_test, core_test_predictions)), 2),
    "minutes",
)
print(
    "Testing R²:",
    round(r2_score(y_core_test, core_test_predictions), 4),
)

In [ ]:
import pandas as pd
import numpy as np

triage_order = [
    "Immediate",
    "Emergency",
    "Urgent",
    "Semi-urgent",
    "Non-urgent",
]

departments = sorted(df["Department"].unique())

priority_test = pd.DataFrame(
    [
        {
            "Department": department,
            "TriageCategory": triage,
        }
        for department in departments
        for triage in triage_order
    ]
)

priority_test["PredictedWaitMinutes"] = core_wait_model.predict(
    priority_test
).round(1)

priority_table = priority_test.pivot(
    index="Department",
    columns="TriageCategory",
    values="PredictedWaitMinutes",
).reindex(columns=triage_order)

display(priority_table)

priority_check = priority_table.apply(
    lambda row: np.all(np.diff(row.to_numpy()) >= 0),
    axis=1,
)

print("PRIORITY ORDER CHECK")
display(priority_check)

print(
    "All departments passed:",
    bool(priority_check.all()),
)

In [ ]:
TRIAGE_QUEUE_WEIGHT = {
    "Immediate": 0.10,
    "Emergency": 0.25,
    "Urgent": 0.50,
    "Semi-urgent": 0.75,
    "Non-urgent": 1.00,
}

def estimate_hybrid_wait(
    department,
    triage_category,
    patients_ahead,
    active_doctors,
    average_consultation_minutes,
    current_doctor_delay_minutes,
    occupancy_rate,
):
    if active_doctors < 1:
        raise ValueError("active_doctors must be at least 1")

    if patients_ahead < 0:
        raise ValueError("patients_ahead cannot be negative")

    patient = pd.DataFrame(
        [{
            "Department": department,
            "TriageCategory": triage_category,
        }]
    )

    ml_base_wait = float(core_wait_model.predict(patient)[0])

    raw_queue_wait = (
        patients_ahead
        * average_consultation_minutes
        / active_doctors
    ) + current_doctor_delay_minutes

    safe_occupancy = float(np.clip(occupancy_rate, 0, 1))

    occupancy_multiplier = 0.85 + (0.50 * safe_occupancy)

    priority_weight = TRIAGE_QUEUE_WEIGHT[triage_category]

    live_queue_wait = (
        raw_queue_wait
        * occupancy_multiplier
        * priority_weight
    )

    final_estimate = (
        0.65 * ml_base_wait
        + 0.35 * live_queue_wait
    )

    return {
        "ML base wait": round(ml_base_wait, 1),
        "Live queue wait": round(live_queue_wait, 1),
        "Final estimated wait": round(max(1, final_estimate), 1),
    }

In [ ]:
low_load = estimate_hybrid_wait(
    department="Cardiology",
    triage_category="Non-urgent",
    patients_ahead=3,
    active_doctors=5,
    average_consultation_minutes=12,
    current_doctor_delay_minutes=0,
    occupancy_rate=0.30,
)

high_load = estimate_hybrid_wait(
    department="Cardiology",
    triage_category="Non-urgent",
    patients_ahead=15,
    active_doctors=2,
    average_consultation_minutes=12,
    current_doctor_delay_minutes=15,
    occupancy_rate=0.90,
)

print("LOW LOAD")
display(low_load)

print("HIGH LOAD")
display(high_load)

difference = (
    high_load["Final estimated wait"]
    - low_load["Final estimated wait"]
)

print("High-load increase:", round(difference, 1), "minutes")

In [ ]:
from sklearn.model_selection import KFold, cross_validate

core_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

core_cv_results = cross_validate(
    core_wait_model,
    X_core,
    y_core,
    cv=core_cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
    n_jobs=-1,
)

core_cv_mae = -core_cv_results["test_mae"]
core_cv_rmse = -core_cv_results["test_rmse"]
core_cv_r2 = core_cv_results["test_r2"]

print("CORE MODEL CROSS-VALIDATION")
print(
    "MAE:",
    round(core_cv_mae.mean(), 2),
    "+/-",
    round(core_cv_mae.std(), 2),
    "minutes",
)
print(
    "RMSE:",
    round(core_cv_rmse.mean(), 2),
    "+/-",
    round(core_cv_rmse.std(), 2),
    "minutes",
)
print(
    "R²:",
    round(core_cv_r2.mean(), 4),
    "+/-",
    round(core_cv_r2.std(), 4),
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import joblib

current_directory = Path.cwd()

if current_directory.name == "ml-notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

model_directory = project_root / "ml-models"
model_directory.mkdir(parents=True, exist_ok=True)

model_path = model_directory / "opd_wait_time_model.joblib"

final_artifact = {
    "artifact_version": "1.0.0",
    "model_name": "SmartCare OPD Base Wait Model",
    "status": "prototype-not-clinically-validated",
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": core_wait_model,
    "features": list(core_features),
    "target": "ActualWaitMinutes",
    "triage_order": list(triage_order),
    "hybrid_config": {
        "ml_base_weight": 0.65,
        "live_queue_weight": 0.35,
        "occupancy_base": 0.85,
        "occupancy_slope": 0.50,
        "triage_queue_weight": TRIAGE_QUEUE_WEIGHT,
    },
    "validation": {
        "cross_validation_folds": 5,
        "mae_mean_minutes": float(core_cv_mae.mean()),
        "mae_std_minutes": float(core_cv_mae.std()),
        "rmse_mean_minutes": float(core_cv_rmse.mean()),
        "rmse_std_minutes": float(core_cv_rmse.std()),
        "r2_mean": float(core_cv_r2.mean()),
        "r2_std": float(core_cv_r2.std()),
    },
    "training_data": {
        "rows": int(len(df)),
        "source_type": "synthetic",
    },
}

joblib.dump(final_artifact, model_path)

loaded_artifact = joblib.load(model_path)

verification_patient = pd.DataFrame(
    [{
        "Department": "Cardiology",
        "TriageCategory": "Non-urgent",
    }]
)

verification_prediction = loaded_artifact["model"].predict(
    verification_patient
)[0]

print("FINAL ARTIFACT SAVED")
print("Path:", model_path.resolve())
print(
    "Size:",
    round(model_path.stat().st_size / (1024 * 1024), 2),
    "MB",
)
print("Status:", loaded_artifact["status"])
print(
    "Verification prediction:",
    round(float(verification_prediction), 1),
    "minutes",
)